# SALES ANALYTICS

In [1]:
import pandas as pd
import numpy as np
import logging
import sys
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv
    

# Load environment variables from .env file
load_dotenv()
# Database connection parameters
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')


In [2]:
# create a connection string
def create_connection_string(user, password, host, port, db_name):
    return f'mysql+mysqlconnector://{user}:{quote_plus(password)}@{host}:{port}/{db_name}'
# create a connection string
connection_string = create_connection_string(DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME)
# create a database engine
engine = create_engine(connection_string)

In [3]:
#loading all tables into dataframes
df_customers = pd.read_sql("SELECT * FROM customers;", engine)
df_dates = pd.read_sql("SELECT * FROM date;", engine)
df_markets = pd.read_sql("SELECT * FROM markets;", engine)
df_products = pd.read_sql("SELECT * FROM products;", engine)
df_transactions = pd.read_sql("SELECT * FROM transactions;", engine)

In [4]:
# function to get table info
def get_table_info(df, table_name):
    print(f"Table: {table_name}")
    print("Table Info:", df.info())
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("Data Types:\n", df.dtypes)
    print("Missing Values:\n", df.isnull().sum())
    print("Duplicates:\n", df.duplicated().sum())
    print("\nSample Data:\n", df.head())
    print("-" * 50)

## Customer Table

In [5]:
#customers table info
get_table_info(df_customers, "customers")
df_customers.head()

Table: customers
<class 'pandas.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_code  38 non-null     str  
 1   custmer_name   38 non-null     str  
 2   customer_type  38 non-null     str  
dtypes: str(3)
memory usage: 1.0 KB
Table Info: None
Shape: (38, 3)
Columns: ['customer_code', 'custmer_name', 'customer_type']
Data Types:
 customer_code    str
custmer_name     str
customer_type    str
dtype: object
Missing Values:
 customer_code    0
custmer_name     0
customer_type    0
dtype: int64
Duplicates:
 0

Sample Data:
   customer_code    custmer_name   customer_type
0        Cus001    Surge Stores  Brick & Mortar
1        Cus002    Nomad Stores  Brick & Mortar
2        Cus003    Excel Stores  Brick & Mortar
3        Cus004  Surface Stores  Brick & Mortar
4        Cus005  Premium Stores  Brick & Mortar
--------------------------------------------------


,customer_code,custmer_name,customer_type
0,Cus001,Surge Stores,Brick & Mortar
1,Cus002,Nomad Stores,Brick & Mortar
2,Cus003,Excel Stores,Brick & Mortar
3,Cus004,Surface Stores,Brick & Mortar
4,Cus005,Premium Stores,Brick & Mortar


In [6]:
df_customers['customer_code'] = df_customers['customer_code'].str.strip().str.upper()
#rename columns
df_customers.rename(columns={'custmer_name': 'customer_name'}, inplace=True)
df_customers.head()

,customer_code,customer_name,customer_type
0,CUS001,Surge Stores,Brick & Mortar
1,CUS002,Nomad Stores,Brick & Mortar
2,CUS003,Excel Stores,Brick & Mortar
3,CUS004,Surface Stores,Brick & Mortar
4,CUS005,Premium Stores,Brick & Mortar


## Dates Table

In [7]:
get_table_info(df_dates, "date")

Table: date
<class 'pandas.DataFrame'>
RangeIndex: 1126 entries, 0 to 1125
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         1126 non-null   object
 1   cy_date      1126 non-null   object
 2   year         1126 non-null   int64 
 3   month_name   1126 non-null   str   
 4   date_yy_mmm  1126 non-null   str   
dtypes: int64(1), object(2), str(2)
memory usage: 44.1+ KB
Table Info: None
Shape: (1126, 5)
Columns: ['date', 'cy_date', 'year', 'month_name', 'date_yy_mmm']
Data Types:
 date           object
cy_date        object
year            int64
month_name        str
date_yy_mmm       str
dtype: object
Missing Values:
 date           0
cy_date        0
year           0
month_name     0
date_yy_mmm    0
dtype: int64
Duplicates:
 0

Sample Data:
          date     cy_date  year month_name date_yy_mmm
0  2017-06-01  2017-06-01  2017       June    17-Jun\r
1  2017-06-02  2017-06-01  2017       June    17-Jun\r


In [8]:
def extract_date_features(df, col, prefix=None):
    df[col] = pd.to_datetime(df[col])
    pre = prefix if prefix else col
    df[f'{pre}_day'] = df[col].dt.day
    df[f'{pre}_day_of_week'] = df[col].dt.dayofweek
    df[f'{pre}_day_name'] = df[col].dt.day_name()
    df[f'{pre}_month'] = df[col].dt.month
    df[f'{pre}_month_name'] = df[col].dt.month_name()
    df[f'{pre}_year'] = df[col].dt.year
    return df

# Extract features for both columns
df_dates = extract_date_features(df_dates, 'date')
df_dates = extract_date_features(df_dates, 'cy_date', prefix='cy')

# Reorder columns
ordered_cols = [
    'date', 'date_day', 'date_day_of_week', 'date_day_name', 'date_month', 'date_month_name', 'date_year',
    'cy_date', 'cy_day', 'cy_day_of_week', 'cy_day_name', 'cy_month', 'cy_month_name', 'cy_year'
]

df_dates = df_dates[ordered_cols]

In [9]:
get_table_info(df_markets, "markets")
df_markets.head()

Table: markets
<class 'pandas.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   markets_code  17 non-null     str  
 1   markets_name  17 non-null     str  
 2   zone          17 non-null     str  
dtypes: str(3)
memory usage: 540.0 bytes
Table Info: None
Shape: (17, 3)
Columns: ['markets_code', 'markets_name', 'zone']
Data Types:
 markets_code    str
markets_name    str
zone            str
dtype: object
Missing Values:
 markets_code    0
markets_name    0
zone            0
dtype: int64
Duplicates:
 0

Sample Data:
   markets_code markets_name     zone
0      Mark001      Chennai    South
1      Mark002       Mumbai  Central
2      Mark003    Ahmedabad    North
3      Mark004    Delhi NCR    North
4      Mark005       Kanpur    North
--------------------------------------------------


,markets_code,markets_name,zone
0,Mark001,Chennai,South
1,Mark002,Mumbai,Central
2,Mark003,Ahmedabad,North
3,Mark004,Delhi NCR,North
4,Mark005,Kanpur,North


In [10]:
# Check if zone has truly empty strings
df_markets[df_markets['zone'].str.strip() == ""]
df_markets['zone'] = df_markets['zone'].replace(r'^\s*$', np.nan, regex=True)
# Replace empty strings or whitespace-only strings with NaN
df_markets['zone'] = df_markets['zone'].replace(r'^\s*$', np.nan, regex=True)
df_markets['zone'] = df_markets['zone'].fillna('Overseas')
df_markets['markets_code'] = df_markets['markets_code'].str.strip().str.upper()

#rename columns
df_markets.rename(columns={'markets_code': 'market_code', "markets_name": "market_name"}, inplace=True)
df_markets

,market_code,market_name,zone
0,MARK001,Chennai,South
1,MARK002,Mumbai,Central
2,MARK003,Ahmedabad,North
3,MARK004,Delhi NCR,North
4,MARK005,Kanpur,North
5,MARK006,Bengaluru,South
6,MARK007,Bhopal,Central
7,MARK008,Lucknow,North
8,MARK009,Patna,North
9,MARK010,Kochi,South


In [11]:
get_table_info(df_products, "products")

Table: products
<class 'pandas.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_code  279 non-null    str  
 1   product_type  279 non-null    str  
dtypes: str(2)
memory usage: 4.5 KB
Table Info: None
Shape: (279, 2)
Columns: ['product_code', 'product_type']
Data Types:
 product_code    str
product_type    str
dtype: object
Missing Values:
 product_code    0
product_type    0
dtype: int64
Duplicates:
 0

Sample Data:
   product_code product_type
0      Prod001  Own Brand\r
1      Prod002  Own Brand\r
2      Prod003  Own Brand\r
3      Prod004  Own Brand\r
4      Prod005  Own Brand\r
--------------------------------------------------


In [12]:
df_products['product_code'] = df_products['product_code'].str.strip().str.upper()
df_products

,product_code,product_type
0,PROD001,Own Brand\r
1,PROD002,Own Brand\r
2,PROD003,Own Brand\r
3,PROD004,Own Brand\r
4,PROD005,Own Brand\r
...,...,...
274,PROD275,Own Brand\r
275,PROD276,Own Brand\r
276,PROD277,Own Brand\r
277,PROD278,Distribution\r


## Transactions

In [13]:
get_table_info(df_transactions, "transactions")

Table: transactions
<class 'pandas.DataFrame'>
RangeIndex: 150283 entries, 0 to 150282
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   product_code   150283 non-null  str    
 1   customer_code  150283 non-null  str    
 2   market_code    150283 non-null  str    
 3   order_date     150283 non-null  object 
 4   sales_qty      150283 non-null  int64  
 5   sales_amount   150283 non-null  float64
 6   currency       150283 non-null  str    
dtypes: float64(1), int64(1), object(1), str(4)
memory usage: 8.0+ MB
Table Info: None
Shape: (150283, 7)
Columns: ['product_code', 'customer_code', 'market_code', 'order_date', 'sales_qty', 'sales_amount', 'currency']
Data Types:
 product_code         str
customer_code        str
market_code          str
order_date        object
sales_qty          int64
sales_amount     float64
currency             str
dtype: object
Missing Values:
 product_code     0
customer_code    0


In [14]:
# cleaning
df_transactions.order_date = pd.to_datetime(df_transactions.order_date)
df_transactions = df_transactions[df_transactions['sales_amount'] > 0]

#conversion rates
usd_to_inr = 82.0
df_transactions.loc[df_transactions['currency'] == 'USD', 'sales_amount'] *= usd_to_inr
df_transactions['currency'] = 'INR'


#clean codes
df_transactions['product_code'] = df_transactions['product_code'].str.strip().str.upper()
df_transactions['customer_code'] = df_transactions['customer_code'].str.strip().str.upper()
df_transactions['market_code'] = df_transactions['market_code'].str.strip().str.upper()